<a href="https://colab.research.google.com/github/Abdoceo/LLMs_Abdo/blob/main/youtube_chat_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install youtube-transcript-api
!pip install -q langchain-openai #embeddings
!pip install chromadb

In [ ]:
import copy

In [ ]:

from langchain_community.document_loaders import YoutubeLoader

Tube_loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?feature=shared&v=6Fa91SY9Gnw"
)
print("YoutubeLoader instance created.")

YoutubeLoader instance created.


In [ ]:
Transcript = Tube_loader.load()
print(Transcript[0].page_content)

According to the TIOBE Index for January 2022,   Python is the programming language with the 
highest increase in ratings in one year,   making it officially the top choice 
of employers and professionals alike.   In fact, it has become a key prerequisite 
to landing a job in data science. Luckily,   Python is also very beginner friendly. So, if 
you’re an aspiring data analyst or data scientist,   learning Python is certainly the way to go.
That said, our job is to make every step of your   learning curve easy and enjoyable. So, If you’re 
still new to Python, you might want to start   for free with our Introduction to Python course. 
We’ve provided a link in the description. But if   you already have some basic Python experience and 
want to take it to the next level, in this video   you’ll discover 4 interesting Python projects 
for beginners – all including code for practice!  Great! So, let’s start with project 
number 1 - Uber Trips Analysis  Since it was founded in 2009, Uber ha

In [ ]:
content_copy = copy.deepcopy(Transcript)
print(content_copy[0].page_content)

According to the TIOBE Index for January 2022,   Python is the programming language with the 
highest increase in ratings in one year,   making it officially the top choice 
of employers and professionals alike.   In fact, it has become a key prerequisite 
to landing a job in data science. Luckily,   Python is also very beginner friendly. So, if 
you’re an aspiring data analyst or data scientist,   learning Python is certainly the way to go.
That said, our job is to make every step of your   learning curve easy and enjoyable. So, If you’re 
still new to Python, you might want to start   for free with our Introduction to Python course. 
We’ve provided a link in the description. But if   you already have some basic Python experience and 
want to take it to the next level, in this video   you’ll discover 4 interesting Python projects 
for beginners – all including code for practice!  Great! So, let’s start with project 
number 1 - Uber Trips Analysis  Since it was founded in 2009, Uber ha

In [ ]:
# ' '.join(content_copy[0].page_content.split())

In [ ]:
for i in content_copy:
  # Clean the text inside each document
  i.page_content = i.page_content.replace(u'\xa0', u' ').replace(u'\uf0a7', u'').replace(u'\n', u'') # u before a string (like u'\xa0') stands for Unicode.
  i.page_content = ' '.join(i.page_content.split())
  transcript_stripped = content_copy[0].page_content
print(transcript_stripped)

According to the TIOBE Index for January 2022, Python is the programming language with the highest increase in ratings in one year, making it officially the top choice of employers and professionals alike. In fact, it has become a key prerequisite to landing a job in data science. Luckily, Python is also very beginner friendly. So, if you’re an aspiring data analyst or data scientist, learning Python is certainly the way to go.That said, our job is to make every step of your learning curve easy and enjoyable. So, If you’re still new to Python, you might want to start for free with our Introduction to Python course. We’ve provided a link in the description. But if you already have some basic Python experience and want to take it to the next level, in this video you’ll discover 4 interesting Python projects for beginners – all including code for practice! Great! So, let’s start with project number 1 - Uber Trips Analysis Since it was founded in 2009, Uber has become one of the most famou

In [ ]:
len(transcript_stripped)

5829

Splitter -- chunking

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
char_splitter = CharacterTextSplitter()

In [ ]:
char_splitter = CharacterTextSplitter(
  separator= '.',
  chunk_size= 500,
  chunk_overlap= 0,
)

In [ ]:
transcript_splitted = char_splitter.split_text(transcript_stripped)
transcript_splitted[1]

'That said, our job is to make every step of your learning curve easy and enjoyable. So, If you’re still new to Python, you might want to start for free with our Introduction to Python course. We’ve provided a link in the description'

In [ ]:
len(transcript_splitted)

15

Embeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings

In [ ]:
import os
from google.colab import userdata

# Retrieve the API key from secrets
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# embedding = OpenAIEmbeddings(model='text-embedding-ada-002')
embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_texts(texts = transcript_splitted,
                                embedding = embedding,
                                persist_directory = "./youtube-agent-project")

In [ ]:
retriever = vectorstore.as_retriever(search_type = 'mmr',
                                   search_kwargs = {'k': 3,
                                   'lambda_mult' : 0.3})

In [ ]:
question = 'could you tell me what this video about?'

In [ ]:
retrived_docs = retriver.invoke(question)

In [ ]:
for i in retrived_docs:
  print(f"{i.page_content}\n")

generate

In [ ]:
# from chromadb.config import System
TEMPLETE_1 = '''
You are helpful  chatbot that answers questions on youtube videos.
anser the questions using only the following cintext:
{context}
'''

message_templete_1 = SystemMessagePromptTemplate.from_template(TEMPLETE_1)

In [ ]:
TEMPLETE_2 = ''' {question} '''

message_templete_2 = HumanMessagePromptTemplate.from_template(TEMPLETE_2)

In [ ]:
chat_templete = ChatPromptTemplate.from_messages([message_templete_1, message_templete_2])

chain

In [ ]:
from langchain_openai.chat_models import ChatOpenAI
from google.colab import userdata

chat = ChatOpenAI(model_name = 'gpt-4',
                  model_kwargs = {'seed':365},
                  max_tokens = 250,
                  temperature = 0,
                  api_key = userdata.get('OPENAI_API_KEY')) # control the level of randomness in responses

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = ( {'context': retriever,
 'question': RunnablePassthrough()}
          | chat_templete
          | chat
          | StrOutputParser()
)

In [ ]:
response = chain.invoke(question)
print(response)